<a href="https://colab.research.google.com/github/engrmaziz/multimodal-cancer-survival-transformer/blob/main/research/Cancer_Survival_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Mount Persistent Storage
from google.colab import drive
import os

# Mount Google Drive to save data permanently
drive.mount('/content/drive')

# Create the repository structure inside your Drive
PROJECT_ROOT = '/content/drive/MyDrive/multimodal-cancer-survival'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')

os.makedirs(DATA_DIR, exist_ok=True)
print(f"✅ Enterprise Storage Volume Mounted: {DATA_DIR}")

In [ ]:
# Cell 2: Query NIH GDC API for Clinical Data
import requests
import json
import pandas as pd

def fetch_tcga_clinical_data(project_id="TCGA-BRCA", max_records=1000):
    print(f"Initiating API request to NIH GDC for {project_id}...")

    endpoint = "https://api.gdc.cancer.gov/cases"

    # 1. Define the query filter (We only want breast cancer patients)
    filters = {
        "op": "in",
        "content": {
            "field": "project.project_id",
            "value": [project_id]
        }
    }

    # 2. Define the exact JSON fields we want the API to return
    # This maps to our Next.js dashboard inputs (Stage, Days to Death, etc.)
    fields = [
        "submitter_id",
        "demographic.vital_status",
        "demographic.days_to_death",
        "diagnoses.tumor_stage",
        "diagnoses.age_at_diagnosis"
    ]

    fields_string = ",".join(fields)

    # 3. Construct the API Payload
    params = {
        "filters": json.dumps(filters),
        "fields": fields_string,
        "format": "JSON",
        "size": max_records
    }

    # 4. Execute POST request
    response = requests.post(endpoint, headers={"Content-Type": "application/json"}, json=params)

    if response.status_code != 200:
        raise Exception(f"API Error {response.status_code}: {response.text}")

    data = response.json()['data']['hits']
    print(f"✅ Successfully retrieved {len(data)} patient records.")

    return data

# Execute the extraction
raw_clinical_json = fetch_tcga_clinical_data(max_records=1098)

In [ ]:
# Cell 3: ETL - Flatten JSON to Tabular Target Matrix
def normalize_clinical_data(raw_json):
    processed_records = []

    for patient in raw_json:
        # Extract demographic data
        demo = patient.get('demographic', {})
        vital_status = demo.get('vital_status', 'Unknown')
        days_to_death = demo.get('days_to_death', None)

        # Extract diagnosis data (Handle lists if multiple diagnoses exist)
        diagnoses = patient.get('diagnoses', [{}])[0]
        stage = diagnoses.get('tumor_stage', 'not reported')
        age_days = diagnoses.get('age_at_diagnosis', None)

        processed_records.append({
            "patient_id": patient.get('submitter_id'),
            "vital_status": vital_status,
            "survival_days": days_to_death,
            "cancer_stage": stage,
            "age_at_diagnosis_years": round(age_days / 365.25, 1) if age_days else None
        })

    df = pd.DataFrame(processed_records)

    # Clean up the targets for our Cox Survival Loss function
    # 1 = Dead (Event Occurred), 0 = Alive (Censored)
    df['event_observed'] = df['vital_status'].apply(lambda x: 1 if x == 'Dead' else 0)

    return df

clinical_df = normalize_clinical_data(raw_clinical_json)

# Save the pristine dataset to your mounted Google Drive
save_path = os.path.join(DATA_DIR, 'tcga_brca_clinical_targets.csv')
clinical_df.to_csv(save_path, index=False)

print(f"✅ Tabular target matrix saved to {save_path}")
print(clinical_df.head())

In [ ]:
# Cell 4: Patch missing survival data and clean stages
def clean_clinical_data(raw_json):
    processed_records = []

    for patient in raw_json:
        demo = patient.get('demographic', {})
        vital_status = demo.get('vital_status', 'Unknown')

        # FIX: Grab both death and follow-up days
        days_to_death = demo.get('days_to_death')
        days_to_follow_up = demo.get('days_to_last_follow_up')

        # Combine into a single survival duration tracking column
        survival_days = days_to_death if vital_status == 'Dead' else days_to_follow_up

        diagnoses = patient.get('diagnoses', [{}])[0]
        stage = diagnoses.get('tumor_stage', 'not reported')
        age_days = diagnoses.get('age_at_diagnosis', None)

        processed_records.append({
            "patient_id": patient.get('submitter_id'),
            "vital_status": vital_status,
            "survival_days": survival_days,
            "cancer_stage": stage,
            "age_at_diagnosis_years": round(age_days / 365.25, 1) if age_days else None,
            "event_observed": 1 if vital_status == 'Dead' else 0
        })

    df = pd.DataFrame(processed_records)

    # Drop patients with absolutely no tracking time
    df = df.dropna(subset=['survival_days'])

    return df

clinical_df = clean_clinical_data(raw_clinical_json)
clinical_df.to_csv(save_path, index=False)

print(f"✅ Cleaned target matrix saved. Valid patient records: {len(clinical_df)}")
print(clinical_df.head())

In [ ]:
# Cell 5: Query and map RNA-Seq file endpoints
import time

def get_rnaseq_file_manifest(project_id="TCGA-BRCA", max_files=500):
    print(f"Querying NIH GDC for {project_id} RNA-Seq Transcripts...")

    endpoint = "https://api.gdc.cancer.gov/files"

    filters = {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.project.project_id", "value": [project_id]}},
            {"op": "in", "content": {"field": "data_category", "value": ["Transcriptome Profiling"]}},
            {"op": "in", "content": {"field": "data_type", "value": ["Gene Expression Quantification"]}}
        ]
    }

    # We need the File ID to download it, and the Case ID to match it to our clinical CSV
    fields = ["file_id", "file_name", "cases.submitter_id"]

    params = {
        "filters": json.dumps(filters),
        "fields": ",".join(fields),
        "format": "JSON",
        "size": max_files # Limiting to 500 for prototype speed
    }

    response = requests.post(endpoint, headers={"Content-Type": "application/json"}, json=params)
    data = response.json()['data']['hits']

    # Flatten the mapping
    manifest = []
    for item in data:
        patient_id = item['cases'][0]['submitter_id']
        manifest.append({
            "patient_id": patient_id,
            "file_id": item['file_id'],
            "file_name": item['file_name']
        })

    df_manifest = pd.DataFrame(manifest)
    print(f"✅ Found {len(df_manifest)} RNA-Seq files.")
    return df_manifest

manifest_df = get_rnaseq_file_manifest(max_files=500) # Testing with 10 patients first
print(manifest_df.head())

In [ ]:
# Cell 6: Download the actual Gene Expression Matrices
import tarfile
import io

RNA_DIR = os.path.join(DATA_DIR, 'rna_seq')
os.makedirs(RNA_DIR, exist_ok=True)

def download_gdc_file(file_id, file_name, save_dir):
    data_endpoint = f"https://api.gdc.cancer.gov/data/{file_id}"
    print(f"Downloading {file_name}...")

    response = requests.get(data_endpoint)

    # Save the unzipped raw file
    file_path = os.path.join(save_dir, file_name)
    with open(file_path, "wb") as f:
        f.write(response.content)

    return file_path

# Download the first patient's matrix as a test
test_patient = manifest_df.iloc[0]
file_path = download_gdc_file(test_patient['file_id'], test_patient['file_name'], RNA_DIR)

# Read the file to see the actual raw genomic numbers
# TCGA files are tab-separated. We skip the first 6 rows (metadata)
rna_matrix = pd.read_csv(file_path, sep='\t', skiprows=6, names=['gene_id', 'unstranded', 'stranded_first', 'stranded_second', 'tpm_unstranded', 'fpkm_unstranded', 'fpkm_uq_unstranded'])

print("\n🧬 ACTUAL RAW GENE MATRIX (Top 5 Genes):")
# We only care about the Gene ID and the TPM (Transcripts Per Million) normalized value
print(rna_matrix[['gene_id', 'tpm_unstranded']].head())

In [ ]:
# Cell 6.5: Batch Download RNA-Seq Matrices
import os
import requests

def batch_download_rna(manifest, save_dir):
    print(f"Starting batch download for {len(manifest)} files...")
    valid_downloads = 0

    for idx, row in manifest.iterrows():
        file_id = row['file_id']
        file_name = row['file_name']
        file_path = os.path.join(save_dir, file_name)

        # Skip if already downloaded to save time and bandwidth
        if os.path.exists(file_path):
            valid_downloads += 1
            continue

        print(f"[{idx+1}/{len(manifest)}] Downloading {file_name}...")
        try:
            response = requests.get(f"https://api.gdc.cancer.gov/data/{file_id}")
            if response.status_code == 200:
                with open(file_path, "wb") as f:
                    f.write(response.content)
                valid_downloads += 1
            else:
                print(f"Failed to download {file_id}: HTTP {response.status_code}")
        except Exception as e:
            print(f"Error downloading {file_id}: {e}")

    print(f"✅ Batch download complete. {valid_downloads}/{len(manifest)} files ready on disk.")

# Execute the batch downloader
batch_download_rna(manifest_df, RNA_DIR)

In [ ]:
# Cell 7: Transform and Filter High-Dimensional Genomic Features
import numpy as np

def process_single_patient_vector(file_path):
    # Load the TSV file skipping metadata
    df = pd.read_csv(file_path, sep='\t', skiprows=6,
                     names=['gene_id', 'gene_name', 'gene_type', 'unstranded',
                            'stranded_first', 'stranded_second', 'tpm_unstranded',
                            'fpkm_unstranded', 'fpkm_uq_unstranded'])

    # Filter for protein-coding genes to eliminate noise from non-coding regions
    coding_df = df[df['gene_type'] == 'protein_coding'].copy()

    # Apply log2(TPM + 1) transformation to handle exponential dynamic range
    coding_df['log2_tpm'] = np.log2(coding_df['tpm_unstranded'] + 1)

    # For a production prototype, let's extract an actionable oncology panel
    # We will sort by expression level to isolate highly active transcripts
    top_expressed = coding_df.sort_values(by='log2_tpm', ascending=False).head(100)

    # Extract the clean numeric array
    feature_vector = top_expressed['log2_tpm'].values
    gene_labels = top_expressed['gene_name'].values

    return feature_vector, gene_labels

# Execute processing on your uploaded file
test_file_path = os.path.join(RNA_DIR, test_patient['file_name'])
matrix_vector, markers = process_single_patient_vector(test_file_path)

print("📊 PROCESS COMPLETE: RAW TSV CONVERTED TO INFERENCE MATRIX")
print(f"Matrix Shape: {matrix_vector.shape} (1D Feature Vector representing top active profiles)")
print(f"First 5 Values of the Matrix: {list(np.round(matrix_vector[:5], 4))}")
print(f"Corresponding Markers: {list(markers[:5])}")

In [ ]:
# Cell 8: Compiling the PyTorch Dataset
import os
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

class TCGASurvivalDataset(Dataset):
    def __init__(self, clinical_csv, rna_dir, manifest_df):
        print("Initializing PyTorch Survival Dataset...")
        self.clinical_df = pd.read_csv(clinical_csv)
        self.rna_dir = rna_dir

        self.features = []
        self.durations = []
        self.events = []
        self.valid_patients = []

        # Iterate through our downloaded manifest
        for _, row in manifest_df.iterrows():
            patient_id = row['patient_id']
            file_name = row['file_name']
            file_path = os.path.join(self.rna_dir, file_name)

            # 1. Match patient to clinical outcomes
            patient_clinical = self.clinical_df[self.clinical_df['patient_id'] == patient_id]
            if patient_clinical.empty:
                continue # Skip if we don't have survival data for this patient

            duration = patient_clinical['survival_days'].values[0]
            event = patient_clinical['event_observed'].values[0]

            # Skip corrupted/missing survival targets
            if pd.isna(duration):
                continue

            # 2. Extract the 100-dimensional feature vector (Reusing our pipeline logic)
            try:
                df = pd.read_csv(file_path, sep='\t', skiprows=6, names=['gene_id', 'gene_name', 'gene_type', 'unstranded', 'stranded_first', 'stranded_second', 'tpm_unstranded', 'fpkm_unstranded', 'fpkm_uq_unstranded'])
                coding_df = df[df['gene_type'] == 'protein_coding'].copy()
                coding_df['log2_tpm'] = np.log2(coding_df['tpm_unstranded'] + 1)
                top_expressed = coding_df.sort_values(by='log2_tpm', ascending=False).head(100)

                feature_vector = top_expressed['log2_tpm'].values.astype(np.float32)

                # Append to our final tensor lists
                self.features.append(feature_vector)
                self.durations.append(float(duration))
                self.events.append(float(event))
                self.valid_patients.append(patient_id)

            except Exception as e:
                print(f"Skipping {patient_id} due to file error: {e}")

        # Convert to PyTorch Tensors
        self.features = torch.tensor(np.array(self.features))
        self.durations = torch.tensor(np.array(self.durations))
        self.events = torch.tensor(np.array(self.events))

        print(f"✅ Dataset Compiled! Total Valid Samples: {len(self.features)}")
        print(f"Features Shape: {self.features.shape}")
        print(f"Durations Shape: {self.durations.shape}")

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.durations[idx], self.events[idx]

# Instantiate the dataset
clinical_csv_path = os.path.join(DATA_DIR, 'tcga_brca_clinical_targets.csv')

# This uses the manifest_df and RNA_DIR we defined in Cells 5 and 6
tcga_dataset = TCGASurvivalDataset(clinical_csv_path, RNA_DIR, manifest_df)

# Create the standard PyTorch DataLoader for batch training
train_loader = DataLoader(tcga_dataset, batch_size=4, shuffle=True)

In [ ]:
# Cell 9: The Multimodal Cross-Attention Architecture
import torch
import torch.nn as nn

class CancerSurvivalTransformer(nn.Module):
    def __init__(self, genomic_features=100, image_embed_dim=384, num_heads=4):
        super().__init__()

        # 1. Genomic Projector: Scales our 100-dim vector to match the 384-dim image space
        self.genomic_proj = nn.Sequential(
            nn.Linear(genomic_features, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, image_embed_dim)
        )

        # 2. Cross-Attention Module
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=image_embed_dim,
            num_heads=num_heads,
            batch_first=True,
            dropout=0.1
        )

        self.norm = nn.LayerNorm(image_embed_dim)

        # 3. Cox Proportional Hazards Risk Layer
        self.risk_predictor = nn.Sequential(
            nn.Linear(image_embed_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1) # Outputs a single continuous Log-Hazard Ratio
        )

    def forward(self, genomic_x, image_x):
        # genomic_x shape: [Batch, 100]
        # image_x shape: [Batch, Num_Patches, 384]

        # Project genomics and add a sequence dimension -> [Batch, 1, 384]
        g_proj = self.genomic_proj(genomic_x).unsqueeze(1)

        # Cross-Attention: Genomics queries the Pathology Images
        attn_out, attn_weights = self.cross_attn(
            query=g_proj,
            key=image_x,
            value=image_x
        )

        # Residual connection + Norm
        fused_vector = self.norm(g_proj + attn_out).squeeze(1) # -> [Batch, 384]

        # Predict Risk
        log_hazard = self.risk_predictor(fused_vector)
        return log_hazard

print("✅ Transformer Architecture Loaded into Memory")

In [ ]:
# Cell 10: Training Loop & Cox Loss Optimization
import torch.optim as optim

def cox_ph_loss(log_hazards, durations, events):
    """Calculates the negative log partial likelihood."""
    # Sort patients by survival time (longest to shortest)
    idx = torch.argsort(durations, descending=True)
    log_hazards = log_hazards[idx]
    events = events[idx]

    # Calculate log-sum-exp of hazards for the risk set
    log_risk_set = torch.log(torch.cumsum(torch.exp(log_hazards), dim=0))

    # Only calculate loss for patients with an observed event (death)
    loss = -torch.sum(events * (log_hazards - log_risk_set)) / (torch.sum(events) + 1e-7)
    return loss

# Initialize Model and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CancerSurvivalTransformer().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

# Training Execution
epochs = 15
print(f"🚀 Starting Training on {device}...")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for gen_batch, dur_batch, event_batch in train_loader:
        gen_batch = gen_batch.to(device)
        dur_batch = dur_batch.to(device)
        event_batch = event_batch.to(device)

        # Mocking the DINOv2 image embeddings to bypass the Colab storage bottleneck
        # Shape: [Batch_Size, 50_Patches, 384_Dimensions]
        mock_image_batch = torch.randn(gen_batch.size(0), 50, 384).to(device)

        optimizer.zero_grad()

        # Forward Pass
        hazards = model(gen_batch, mock_image_batch).squeeze()

        # Calculate Loss
        loss = cox_ph_loss(hazards, dur_batch, event_batch)

        # Backpropagation
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] | Cox Survival Loss: {avg_loss:.4f}")

# Save the trained enterprise weights
torch.save(model.state_dict(), os.path.join(PROJECT_ROOT, "multimodal_survival_weights.pth"))
print("✅ Training Complete. Model weights saved for FastAPI deployment!")